In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture

In [2]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[y_true[i]][y_pred[i]] += 1

    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [3]:
filenames = [0,2,3,4,5]

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

# pd.set_option('future.no_silent_downcasting', True)

[0, 2, 3, 4, 5]

In [8]:
exp_val = []
y_true_all_exp_val = []
for i in range(len(filenames)):
    exp_val.append(pd.read_csv(f'val_{filenames[i]}_GMM_BIC_1_10_scores_corr.csv'))
    y_true_all_exp_val.append(exp_val[i]['Label'].values.tolist())
    exp_val[i] = exp_val[i].drop(columns=['Label'])

In [9]:
exp_val[0]

,0,1,2,3,4,5
0,NaN,19.935705,-48.818030,-15808.303205,-21.500241,-65.854462
1,NaN,-8.412854,29.685671,17.478867,-320.092369,-158.010622
2,NaN,24.235509,-36.801325,-14968.220023,-20.034179,-37.640116
3,NaN,-102.669033,-52.032445,26.650458,-974.547268,-195.489090
4,NaN,27.129415,-17.689401,-15496.922456,-15.503954,-0.599197
...,...,...,...,...,...,...
519951,NaN,24.848032,-30.161328,-15479.226154,-18.381144,-26.695069
519952,NaN,4.697448,-172.079922,-35874.640927,27.759064,-18.744434
519953,NaN,18.034682,-8.887314,-17512.429612,-23.359146,-1.604396
519954,NaN,26.376891,-4.701878,-14893.066887,-15.588237,8.495412


In [14]:
from sklearn.metrics import classification_report
ths = [15, 16, 17, 18, 19, 20, 20.5, 21]
f1s = [[],[],[],[],[],[],[],[]]
for i in range(len(exp_val)):
    index = 0
    for th in ths:
        y_pred = []
        prog= 0
        for j in range(len(exp_val[i])):
            # print(exp[i][j])
            m = np.nanmax(exp_val[i].values[j])
            # print(m)
            if m > th:
                y_pred.append(exp_val[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)
        # print(y_pred[:10])

        cm, cm_porcento = matriz_conf(y_true_all_exp_val[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][filenames[i]]
        fp = cm[-1].sum() - tp
        fn = cm.iloc[filenames[i]].sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_val[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))

[0, 1, 2, 3, 4, 5, -1]
th = 15 hidden = 0


,0,1,2,3,4,5,-1
0,0,122107,0,0,0,0,80121
1,0,122748,16,1,90,380,5301
2,0,0,82042,0,0,0,263
3,0,0,3,60944,0,0,3
4,0,39,0,0,44698,16,1037
5,0,5,0,0,9,122,11
-1,0,0,0,0,0,0,0


th: 15 hidden: 0 acc:0.752436744647624 recall:0.3961914274976759 precision:0.9237340896513558 f1:0.5545396658407276
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.92      0.40      0.55    202228
           1       0.50      0.95      0.66    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.98      0.99     45790
           5       0.24      0.83      0.37       147

    accuracy                           0.75    519956
   macro avg       0.78      0.86      0.76    519956
weighted avg       0.85      0.75      0.74    519956

[0, 1, 2, 3, 4, 5, -1]
th = 16 hidden = 0


,0,1,2,3,4,5,-1
0,0,112905,0,0,0,0,89323
1,0,120876,16,1,90,374,7179
2,0,0,81819,0,0,0,486
3,0,0,3,60933,0,0,14
4,0,38,0,0,44691,16,1045
5,0,5,0,0,9,122,11
-1,0,0,0,0,0,0,0


th: 16 hidden: 0 acc:0.7660571279108233 recall:0.44169452301362816 precision:0.9109200677150258 f1:0.5949195100670693
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.91      0.44      0.59    202228
           1       0.52      0.94      0.67    128536
           2       1.00      0.99      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.98      0.99     45790
           5       0.24      0.83      0.37       147

    accuracy                           0.76    519956
   macro avg       0.78      0.86      0.77    519956
weighted avg       0.85      0.76      0.76    519956

[0, 1, 2, 3, 4, 5, -1]
th = 17 hidden = 0


...

th: 20 hidden: 5 acc:0.9398776050281178 recall:0.6054421768707483 precision:0.0028441774255400743 f1:0.005661757689493941
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.61      0.01       147
           0       1.00      0.99      1.00    202228
           1       1.00      0.77      0.87    128536
           2       1.00      0.99      0.99     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.99      0.99     45790

    accuracy                           0.94    519956
   macro avg       0.83      0.89      0.81    519956
weighted avg       1.00      0.94      0.97    519956

[0, 1, 2, 3, 4, 5, -1]
th = 20.5 hidden = 5


,0,1,2,3,4,5,-1
0,200901,55,0,0,0,0,1272
1,97,96289,13,0,87,0,32050
2,0,0,81274,0,0,0,1031
3,0,0,3,60821,0,0,126
4,0,8,0,0,45245,0,537
5,0,15,0,0,7,0,125
-1,0,0,0,0,0,0,0


th: 20.5 hidden: 5 acc:0.9326135288370554 recall:0.8503401360544217 precision:0.0035570985458581146 f1:0.0070845613239628215
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.85      0.01       147
           0       1.00      0.99      1.00    202228
           1       1.00      0.75      0.86    128536
           2       1.00      0.99      0.99     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.99      0.99     45790

    accuracy                           0.93    519956
   macro avg       0.83      0.93      0.81    519956
weighted avg       1.00      0.93      0.96    519956

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 5


,0,1,2,3,4,5,-1
0,200655,22,0,0,0,0,1551
1,50,93150,13,0,85,0,35238
2,0,0,81081,0,0,0,1224
3,0,0,3,60821,0,0,126
4,0,8,0,0,45070,0,712
5,0,6,0,0,6,0,135
-1,0,0,0,0,0,0,0


th: 21 hidden: 5 acc:0.9252571371423736 recall:0.9183673469387755 precision:0.0034627815113117528 f1:0.006899547696317686
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.92      0.01       147
           0       1.00      0.99      1.00    202228
           1       1.00      0.72      0.84    128536
           2       1.00      0.99      0.99     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.98      0.99     45790

    accuracy                           0.92    519956
   macro avg       0.83      0.93      0.80    519956
weighted avg       1.00      0.92      0.96    519956



# Média absolute threshold

In [15]:
for i in range(len(f1s)):
    print(f'Média f1 absolute th {ths[i]}: {np.mean(np.array(f1s[i]))}')

Média f1 absolute th 15: 0.48636407718066854
Média f1 absolute th 16: 0.48733332656136297
Média f1 absolute th 17: 0.48885414426759455
Média f1 absolute th 18: 0.48548241940464376
Média f1 absolute th 19: 0.585704233560568
Média f1 absolute th 20: 0.582632204996382
Média f1 absolute th 20.5: 0.571187890707439
Média f1 absolute th 21: 0.5503277923828185
